## Q3:

This visualization also consists of small multiples. 9 Choropleth maps. They show, over groups of three years, how the quantity of fires varies over time in each _comarca_.    The cholor palette shows how the absolute number of fires in each time period deviates from the median in that region.

A choropleth was chosen because we thought of the importance of distinguishing by region. The colors also play an important role, the main reason for which we didn't color the relative change instead, is because we wanted to give more importance to those _comarques_ with a higher number of fires overall, and using a relative scale would give the same "importance" to a change from 2 to 3 fires than 20 to 30, for example.

This visualization can be complemented with the small multiples above it, which can show the total number of fires each year, by looking at the total aggregate over all causes.

In [ ]:
# -*- coding: utf-8 -*-
"""
Incendis forestals a Catalunya (1998-2024) — Visualitzacions Finals Q1-Q4
Information Visualization | Projecte 1
Inspiració d'estil: Periodisme de dades (Roger Tugas)
Tots els fitxers generats acaben amb el sufix '_vg.html'.
"""

import copy
import json
import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# %% ===================================================================
# 0. CÀRREGA I NETEJA DE DADES
# =======================================================================
DATA_PATH = "data/incendis_final.csv"
COMARQUES_MAP_PATH = 'data/divisions-administratives-v2r2-comarques-500000-20260120.json'

df = pd.read_csv(DATA_PATH)
top_causes = df['Causa'].value_counts().nlargest(6).index
df['CausaAgrupada'] = df['Causa'].where(df['Causa'].isin(top_causes), 'Altres')
causa_domain = sorted(df['CausaAgrupada'].unique())
causa_scale = alt.Scale(domain=causa_domain, scheme='dark2')

 


# --- GeoJSON i càlcul d'aspect ratio real ---
geojson_path = (
    COMARQUES_MAP_PATH
)
with open(geojson_path, encoding='utf-8') as f:
    geo_dict = json.load(f)
comarques_geo = alt.Data(values=geo_dict['features'])
noms_geojson = {feat['properties']['NOMCOMAR'] for feat in geo_dict['features']}


def _calcular_bbox(geo_d):
    xs, ys = [], []

    def recorre(coords):
        if isinstance(coords[0], (int, float)):
            xs.append(coords[0])
            ys.append(coords[1])
        else:
            for c in coords:
                recorre(c)

    for feat in geo_d['features']:
        recorre(feat['geometry']['coordinates'])
    return min(xs), max(xs), min(ys), max(ys)


_minx, _maxx, _miny, _maxy = _calcular_bbox(geo_dict)
ASPECT_RATIO = (_maxx - _minx) / (_maxy - _miny)


def mida_mapa(amplada):
    return amplada, round(amplada / ASPECT_RATIO)


PROJECCIO = 'identity'
REFLECTEIX_Y = True


def titol_rt(titol, subtitol):
    return alt.TitleParams(
        text=titol,
        subtitle=subtitol,
        anchor='start',
        fontSize=20,
        fontWeight='bold',
        subtitleFontSize=13,
        subtitleColor='#555555',
        offset=12
    )

In [14]:
# %% ===================================================================
# Q3 · 9 Mapes (1r amb escala absoluta pròpia + 8 amb diferència absoluta vs. trienni anterior)
# =======================================================================
df_q3_comarca = (
    df.groupby(['PeriodeQ3', 'Nom comarca'], as_index=False, observed=True)
    .size().rename(columns={'size': 'Nombre_Incendis', 'PeriodeQ3': 'Periode'})
)
totes_comarques = sorted(list(noms_geojson))
index_complet_q3 = pd.MultiIndex.from_product([labels_q3, totes_comarques], names=['Periode', 'Nom comarca'])
df_q3_comarca = (
    df_q3_comarca.set_index(['Periode', 'Nom comarca'])
    .reindex(index_complet_q3, fill_value=0).reset_index()
)

df_q3_comarca['Mitjana_focs_hist'] = df_q3_comarca.groupby('Nom comarca')['Nombre_Incendis'].transform("median")

df_q3_comarca['Dif_Mitjana_focs'] = df_q3_comarca['Nombre_Incendis'] - df_q3_comarca['Mitjana_focs_hist']

max_abs_p0 = float(df_q3_comarca[df_q3_comarca['Periode'] == labels_q3[0]]['Nombre_Incendis'].max())
lim_dif_ant = float(np.nanpercentile(df_q3_comarca['Dif_Mitjana_focs'].abs(), 96))

w3, h3 = mida_mapa(160)
mapes_q3 = []

for idx_m, p in enumerate(labels_q3):
    dades_p = df_q3_comarca[df_q3_comarca['Periode'] == p]
    p_ant = labels_q3[idx_m - 1]
    mostra_leg_dif = (idx_m == len(labels_q3) - 1)
    submap = alt.Chart(comarques_geo).mark_geoshape(stroke='white', strokeWidth=0.6).transform_lookup(
        lookup='properties.NOMCOMAR',
        from_=alt.LookupData(
            dades_p, 'Nom comarca',
            ['Nombre_Incendis', 'Mitjana_focs_hist', 'Dif_Mitjana_focs']
        )
    ).encode(
        color=alt.Color(
            'Dif_Mitjana_focs:Q',
            scale=alt.Scale(
                domain=[-lim_dif_ant, 0, lim_dif_ant],
                range=['#542788', '#f7f7f7', '#e66101'],
                clamp=True
            ),
            title="Variació vs. mitjana històrica",
            legend=alt.Legend(orient='bottom', gradientLength=200, titleFontSize=11, labelFontSize=10)
            if mostra_leg_dif else None
        ),
        tooltip=[
            alt.Tooltip('properties.NOMCOMAR:N', title='Comarca'),
            alt.Tooltip('Nombre_Incendis:Q', title=f"Incendis ({p})"),
            alt.Tooltip("Mitjana_focs_hist:Q", title="Mitjana històrica", format='.2f')
            # alt.Tooltip('Incendis_Anterior:Q', title=f"Incendis trienni previ ({p_ant})", format=','),
            # alt.Tooltip('Dif_Mitjana_focs:Q', title=f"Diferència ({p} vs {p_ant})", format='+,d')
        ]
    ).project(type=PROJECCIO, reflectY=REFLECTEIX_Y).properties(
        width=w3, height=h3,
        title=alt.TitleParams(f"{p}", fontSize=14, fontWeight='bold'))

    mapes_q3.append(submap)

q3_final = alt.concat(*mapes_q3, columns=9, spacing=12).resolve_scale(color='independent').properties(
    title=titol_rt(
        "Q3 · Evolució triennal per comarca: variació absoluta respecte la mediana històrica de focs (1998-2024)",
        ""
    ),
    ).configure_view(stroke=None)

q3_final
q3_final.save('outputs/Q3.json')
q3_final.save('outputs/Q3.html')


Matriu 3x3

In [15]:
w3, h3 = mida_mapa(275)

for m in mapes_q3:
    m.width = w3
    m.height = h3

q3_final_v2 = alt.concat(*mapes_q3, columns=3, spacing=10).resolve_scale(color='independent').properties(
    title=titol_rt(
        "Q3 · Evolució triennal per comarca: variació absoluta respecte la mediana històrica de focs (1998-2024)",
        ""
    ),
).configure_view(stroke=None)

q3_final_v2
q3_final.save('outputs/Q3_3x3.json')
q3_final_v2.save('outputs/Q3_3x3.html')